# Order Book Viewer

Depth-5 L2 order-book view using `tools.lob_viewer.plot_order_book`, L2 depth parquet files in `data/orderbook_l2_parquet`, and L3/MBO event parquet files in `data/databento_glbx_mdp3_mbo_full_day_parquet`. A separate feature/return parquet can supply arbitrary overlay columns, and an optional fitted `Pipeline` can add predictions from that same file.

The defaults use the ES regular-hours window. Styling follows common trading-screen convention: dark background, bid levels in green, ask levels in red/orange, small cross markers at every displayed book update, and L3 events as directional markers.


In [1]:
from pathlib import Path
import sys

import importlib

import polars as pl
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "tools").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import tools.viewer as viewer_mod
import tools.lob_viewer as lob_viewer_mod
importlib.reload(viewer_mod)
importlib.reload(lob_viewer_mod)
from tools.data import DateFrame, Raw
from tools.lob_viewer import INVALID_PRICE, PRICE_SCALE, plot_order_book
from tools.pipeline import Pipeline
from tools.viewer import add_x_pan_buttons


In [2]:
try:
    import plotly  # noqa: F401
    import plotly.io as pio
    pio.renderers.default = "plotly_mimetype+notebook_connected"
except ImportError as exc:
    raise RuntimeError(
        "This notebook needs Plotly because tools.viewer imports it lazily when rendering. "
        "Install plotly in the active notebook environment, then rerun."
    ) from exc

## Window And Instrument

`plot_order_book(timezone=...)` interprets naive `start`/`end` strings in the display timezone, then filters the UTC source column efficiently before rendering.


In [3]:
L2_DIR = PROJECT_ROOT / "data" / "orderbook_l2_parquet"
L3_DIR = PROJECT_ROOT / "data" / "databento_glbx_mdp3_mbo_full_day_parquet"
FEATURE_DIR = PROJECT_ROOT / "data" / "orderbook_feature_return_parquet"

PRODUCT = "ESM6"
SESSION_DATE = "2026-05-28"
DISPLAY_TZ = "America/New_York"

# START = f"{SESSION_DATE} 10:11:00"
# END = f"{SESSION_DATE} 10:13:00"
START = f"{SESSION_DATE} 09:30:00"
END = f"{SESSION_DATE} 16:00:00"
DEPTH_LEVELS = 1
MAX_POINTS_PER_SERIES = 15_000

# Independent feature/return overlay source. Set an exact path to avoid auto-discovery;
# relative paths are resolved from PROJECT_ROOT. Choose any numeric columns to plot.
FEATURE_RETURN_PATH = None
FEATURE_RETURN_COLS = ("forward_mid_return_bps",)  # add features such as "imb_d1"
SIGNAL_TITLE = "prediction, forward return, and cumulative scores"
SIGNAL_YAXIS_TITLE = "bps"
SIGNAL_VALUE_LABEL = "bps"

# Saved pipeline. Date and hash prefixes must select one run.
LOAD_PIPELINE = True
PIPELINE_ROOT = PROJECT_ROOT / "notebooks" / "dump"
PIPELINE_DATE = "20260709T0250"
PIPELINE_HASH = "5e338005"
# PIPELINE_DATE = "20260711T0304"
# PIPELINE_HASH = "8d99fda3"
PREDICTION_COLS = ("prediction_q50",)
# Empty filters intentionally include overnight rows instead of the saved RTH test filter.
PREDICTION_FILTERS = ()

# Running normalized scores over the selected START/END window.
CUMULATIVE_SCORES = {
    "cumulative_unit_pnl": {
        "kind": "unit_pnl",
        "prediction_col": "prediction_q50",
        "threshold": 1.0,
        "power": 0,
    },
    # "cumulative_quantile_pnl": {
    #     "kind": "quantile_pnl",
    #     "buy_col": "prediction_q50",
    #     "sell_col": "prediction_q90",
    #     "thd_buy": -1.0,
    #     "thd_sell": 1.0,
    #     "power": 0,
    # },
}

l2_matches = sorted(L2_DIR.glob(f"{PRODUCT}_{SESSION_DATE}_*_l2_d5.parquet"))
if not l2_matches:
    raise FileNotFoundError(f"No L2 parquet matched {PRODUCT}_{SESSION_DATE}_*_l2_d5.parquet in {L2_DIR}")
BOOK_PATH = l2_matches[0]

l3_matches = sorted(L3_DIR.glob(f"{PRODUCT}_{SESSION_DATE}_*_full_day.parquet"))
if not l3_matches:
    raise FileNotFoundError(f"No L3 parquet matched {PRODUCT}_{SESSION_DATE}_*_full_day.parquet in {L3_DIR}")
L3_PATH = l3_matches[0]

BOOK_PATH, L3_PATH


(PosixPath('/home/jli/projects/rep/data/orderbook_l2_parquet/ESM6_2026-05-28_extra_normal_es_full_day_l2_d5.parquet'),
 PosixPath('/home/jli/projects/rep/data/databento_glbx_mdp3_mbo_full_day_parquet/ESM6_2026-05-28_extra_normal_es_full_day.parquet'))

## Saved Pipeline

`regular_loader` reads the precomputed feature/return files with `Raw.load_date`. `Pipeline.from_saved` reconstructs the selected saved pipeline directly from its manifest, model, transform, and filters. `PIPELINE_DATE` and `PIPELINE_HASH` may be full values or unique prefixes; they must resolve to exactly one run. Set `LOAD_PIPELINE=False` for feature/return overlays without predictions.


In [4]:
PIPELINE_PROD = PRODUCT[:2]
FEATURE_RETURN_PATTERN = str(
    FEATURE_DIR
    / "{prod}M6_{d}_{tag}_{prod_s}_full_day_l2_d5_features_return.parquet"
)
LOAD_COLS = None


def load_feature_return_date(day: str, prod: str = PIPELINE_PROD) -> DateFrame:
    return Raw.load_date(day, prod, path=FEATURE_RETURN_PATTERN, cols=LOAD_COLS)


def regular_loader(dates: list[str]) -> list[DateFrame]:
    return [load_feature_return_date(day) for day in dates]


PIPELINE = (
    Pipeline.from_saved(
        PIPELINE_ROOT,
        stamp=PIPELINE_DATE,
        run_hash=PIPELINE_HASH,
        data_loader=regular_loader,
    )
    if LOAD_PIPELINE
    else None
)

if PIPELINE is not None:
    schema_path, _ = Raw.resolve_path(
        SESSION_DATE, PIPELINE_PROD, FEATURE_RETURN_PATTERN
    )
    feature_schema = pl.scan_parquet(schema_path).collect_schema()
    meta_cols = [
        col
        for col in feature_schema.names()
        if col not in PIPELINE.features
        and col != PIPELINE.target
        and not col.startswith("ewma_var")
    ]
    LOAD_COLS = list(
        dict.fromkeys(
            [*meta_cols, *PIPELINE.features, PIPELINE.target, "ewma_var_hl120s"]
        )
    )

display(
    {
        "pipeline_date": PIPELINE_DATE if PIPELINE else None,
        "pipeline_hash": PIPELINE_HASH if PIPELINE else None,
        "pipeline_adapter": type(PIPELINE.adapter).__name__ if PIPELINE else None,
        "pipeline_features": len(PIPELINE.features) if PIPELINE else 0,
        "data_loader": regular_loader.__name__,
    }
)


{'pipeline_date': '20260709T0250',
 'pipeline_hash': '5e338005',
 'pipeline_adapter': 'TorchAdapter',
 'pipeline_features': 34,
 'data_loader': 'regular_loader'}

In [5]:
raw_l2 = pl.scan_parquet(BOOK_PATH)
raw_l3 = pl.scan_parquet(L3_PATH)

{
    "l2": raw_l2.collect_schema(),
    "l3": raw_l3.collect_schema(),
}


{'l2': Schema([('ts_event', Datetime(time_unit='ns', time_zone='UTC')),
         ('row_nr', UInt64),
         ('sequence', UInt32),
         ('publisher_id', UInt16),
         ('instrument_id', UInt32),
         ('trade_px', Int64),
         ('trade_sz', UInt32),
         ('trade_side', UInt8),
         ('trade_px_last', Int64),
         ('trade_vwap', Float64),
         ('trade_levels', UInt32),
         ('trade_fills', UInt32),
         ('trade_posted_sz', UInt32),
         ('bid_px_0', Int64),
         ('bid_sz_0', UInt64),
         ('bid_ct_0', UInt32),
         ('ask_px_0', Int64),
         ('ask_sz_0', UInt64),
         ('ask_ct_0', UInt32),
         ('bid_px_1', Int64),
         ('bid_sz_1', UInt64),
         ('bid_ct_1', UInt32),
         ('ask_px_1', Int64),
         ('ask_sz_1', UInt64),
         ('ask_ct_1', UInt32),
         ('bid_px_2', Int64),
         ('bid_sz_2', UInt64),
         ('bid_ct_2', UInt32),
         ('ask_px_2', Int64),
         ('ask_sz_2', UInt64),
       

## Lazy Source

Databento-style prices are fixed-point integers. `plot_order_book` keeps raw L2/L3 parquet columns as hover data and decodes prices only for display.


In [6]:
def valid_price(px_col: str, sz_col: str | None = None) -> pl.Expr:
    valid_px = (pl.col(px_col) > 0) & (pl.col(px_col) < INVALID_PRICE)
    if sz_col is not None:
        valid_px = valid_px & (pl.col(sz_col) > 0)
    return valid_px


def display_price(px_col: str, sz_col: str | None = None) -> pl.Expr:
    return pl.when(valid_price(px_col, sz_col)).then(pl.col(px_col) / PRICE_SCALE).otherwise(None)


book = pl.scan_parquet(BOOK_PATH)
l3 = pl.scan_parquet(L3_PATH)


In [7]:
session_year, session_month, session_day = map(int, SESSION_DATE.split("-"))

l2_summary = book.filter(
    pl.col("ts_event").dt.convert_time_zone(DISPLAY_TZ).is_between(
        pl.datetime(session_year, session_month, session_day, 9, 30, time_zone=DISPLAY_TZ),
        pl.datetime(session_year, session_month, session_day, 9, 30, 5, time_zone=DISPLAY_TZ),
    )
).select(
    pl.len().alias("rows"),
    valid_price("trade_px", "trade_sz").sum().alias("trade_rows"),
    valid_price("bid_px_0", "bid_sz_0").sum().alias("valid_l1_bid_rows"),
    valid_price("ask_px_0", "ask_sz_0").sum().alias("valid_l1_ask_rows"),
    display_price("bid_px_0", "bid_sz_0").median().alias("median_bid"),
    display_price("ask_px_0", "ask_sz_0").median().alias("median_ask"),
).collect()

l3_summary = (
    l3.filter(
        pl.col("ts_event").dt.convert_time_zone(DISPLAY_TZ).is_between(
            pl.datetime(session_year, session_month, session_day, 9, 30, time_zone=DISPLAY_TZ),
            pl.datetime(session_year, session_month, session_day, 9, 30, 5, time_zone=DISPLAY_TZ),
        )
    )
    .group_by(["action", "side"])
    .agg(pl.len().alias("rows"))
    .sort(["action", "side"])
    .collect()
)

display(l2_summary)
display(l3_summary)


rows,trade_rows,valid_l1_bid_rows,valid_l1_ask_rows,median_bid,median_ask
u32,u32,u32,u32,f64,f64
10029,891,10029,10029,7535.5,7535.75


action,side,rows
str,str,u32
"""A""","""A""",2724
"""A""","""B""",3071
"""C""","""A""",2730
"""C""","""B""",2866
"""F""","""A""",931
"""F""","""B""",1092
"""M""","""A""",693
"""M""","""B""",813
"""T""","""A""",475


## Optional Feature, Return, And Pipeline Overlays

`FEATURE_RETURN_PATH` is independent of the L2/L3 files. Select any numeric feature or return columns with `FEATURE_RETURN_COLS`; they can be plotted without a pipeline. When `LOAD_PIPELINE=True`, the saved pipeline loaded above supplies predictions from its model and fitted transform. The viewer never refits the pipeline. `CUMULATIVE_SCORES` adds running normalized `unit_pnl` or `quantile_pnl` lines over the selected window; these are overlapping-forward-return model scores, not an executable trading equity curve. Predictions and selected source columns retain exact timestamp/row alignment.


In [8]:
def add_cumulative_scores(frame, target_col, configs):
    if not configs:
        return frame, ()
    order_cols = [
        col for col in ("ts_event", "row_nr", "sequence") if col in frame.columns
    ]
    frame = frame.sort(order_cols) if order_cols else frame
    output_cols = []

    for idx, (output_col, spec) in enumerate(configs.items()):
        if output_col in frame.columns:
            raise ValueError(f"Cumulative score column already exists: {output_col}")
        kind = spec.get("kind")
        power = int(spec.get("power", 0))
        if power < 0:
            raise ValueError(f"{output_col}: power must be non-negative")

        if kind == "unit_pnl":
            prediction_col = spec["prediction_col"]
            required = (target_col, prediction_col)
            prediction = pl.col(prediction_col).cast(pl.Float64)
            valid = pl.all_horizontal(
                [
                    pl.col(col).is_not_null() & pl.col(col).is_finite()
                    for col in required
                ]
            )
            active = valid & (prediction.abs() > float(spec.get("threshold", 0.0)))
            side = prediction.sign()
            signal = prediction
        elif kind == "quantile_pnl":
            buy_col = spec["buy_col"]
            sell_col = spec["sell_col"]
            required = (target_col, buy_col, sell_col)
            buy_prediction = pl.col(buy_col).cast(pl.Float64)
            sell_prediction = pl.col(sell_col).cast(pl.Float64)
            valid = pl.all_horizontal(
                [
                    pl.col(col).is_not_null() & pl.col(col).is_finite()
                    for col in required
                ]
            )
            raw_buy = valid & (buy_prediction > float(spec.get("thd_buy", 0.0)))
            raw_sell = valid & (sell_prediction < float(spec.get("thd_sell", 0.0)))
            overlap = raw_buy & raw_sell
            buy = raw_buy & ~overlap
            sell = raw_sell & ~overlap
            active = buy | sell
            side = pl.when(buy).then(1.0).when(sell).then(-1.0).otherwise(0.0)
            signal = (
                pl.when(buy)
                .then(buy_prediction)
                .when(sell)
                .then(sell_prediction)
                .otherwise(0.0)
            )
        else:
            raise ValueError(f"{output_col}: unknown cumulative score kind {kind!r}")

        missing = [col for col in required if col not in frame.columns]
        if missing:
            raise ValueError(f"{output_col}: missing columns {missing}")
        weight = signal.abs().pow(power) if power else pl.lit(1.0)
        pnl_part = f"__score_{idx}_pnl_part"
        norm_part = f"__score_{idx}_norm_part"
        cumulative_pnl = f"__score_{idx}_cumulative_pnl"
        cumulative_norm = f"__score_{idx}_cumulative_norm"
        frame = (
            frame.with_columns(
                pl.when(active)
                .then(side * weight * pl.col(target_col).cast(pl.Float64))
                .otherwise(0.0)
                .alias(pnl_part),
                pl.when(active).then(weight).otherwise(0.0).alias(norm_part),
            )
            .with_columns(
                pl.col(pnl_part).cum_sum().alias(cumulative_pnl),
                pl.col(norm_part).cum_sum().alias(cumulative_norm),
            )
            .with_columns(
                pl.when(pl.col(cumulative_norm) > 0.0)
                .then(pl.col(cumulative_pnl) / pl.col(cumulative_norm))
                .otherwise(0.0)
                .alias(output_col)
            )
            .drop(pnl_part, norm_part, cumulative_pnl, cumulative_norm)
        )
        output_cols.append(output_col)

    return frame, tuple(output_cols)


signal_source = None
selected_signal_cols = ()
selected_prediction_cols = ()
cumulative_score_cols = ()
feature_return_cols = (
    (FEATURE_RETURN_COLS,)
    if isinstance(FEATURE_RETURN_COLS, str)
    else tuple(FEATURE_RETURN_COLS)
)

feature_path = None
if FEATURE_RETURN_PATH is not None:
    feature_path = Path(FEATURE_RETURN_PATH)
    if not feature_path.is_absolute():
        feature_path = PROJECT_ROOT / feature_path
elif feature_return_cols or PIPELINE is not None:
    feature_matches = sorted(
        FEATURE_DIR.glob(f"{PRODUCT}_{SESSION_DATE}_*_features_return.parquet")
    )
    if not feature_matches:
        raise FileNotFoundError(
            f"No feature parquet matched {PRODUCT}_{SESSION_DATE}_*_features_return.parquet "
            f"in {FEATURE_DIR}; set FEATURE_RETURN_PATH explicitly"
        )
    feature_path = feature_matches[0]

feature_frame = None
if feature_path is not None:
    if not feature_path.exists():
        raise FileNotFoundError(f"Feature/return parquet does not exist: {feature_path}")
    feature_frame = pl.scan_parquet(feature_path)
    feature_schema = feature_frame.collect_schema()
    missing_feature_cols = [
        col for col in feature_return_cols if col not in feature_schema.names()
    ]
    if missing_feature_cols:
        raise ValueError(f"Feature/return columns not found: {missing_feature_cols}")

if PIPELINE is not None:
    signal_source = PIPELINE.predict_frame(
        dates=[SESSION_DATE],
        feature_source=feature_path,
        key_cols=("ts_event", "row_nr", "sequence", *feature_return_cols),
        filters=PREDICTION_FILTERS,
        start=START,
        end=END,
        timezone=DISPLAY_TZ,
    )
    available_prediction_cols = tuple(
        col for col in signal_source.columns if col.startswith("prediction")
    )
    if PREDICTION_COLS is None:
        if "prediction_q50" in available_prediction_cols:
            selected_prediction_cols = ("prediction_q50",)
        elif "prediction" in available_prediction_cols:
            selected_prediction_cols = ("prediction",)
        else:
            selected_prediction_cols = available_prediction_cols
    elif isinstance(PREDICTION_COLS, str):
        selected_prediction_cols = (PREDICTION_COLS,)
    else:
        selected_prediction_cols = tuple(PREDICTION_COLS)
    missing_prediction_cols = [
        col for col in selected_prediction_cols if col not in signal_source.columns
    ]
    if missing_prediction_cols:
        raise ValueError(f"Prediction columns not found: {missing_prediction_cols}")
    if not selected_prediction_cols:
        raise ValueError("Pipeline produced no prediction columns")
    signal_source, cumulative_score_cols = add_cumulative_scores(
        signal_source, PIPELINE.target, CUMULATIVE_SCORES
    )
elif feature_frame is not None and feature_return_cols:
    signal_source = feature_frame

selected_signal_cols = tuple(
    dict.fromkeys(
        (*feature_return_cols, *selected_prediction_cols, *cumulative_score_cols)
    )
)
if feature_path is None:
    print("No feature/return source selected; plotting order-book data only.")
elif not selected_signal_cols:
    print("Feature/return file selected, but no overlay columns were requested.")

display(
    {
        "feature_return_path": feature_path,
        "feature_return_cols": feature_return_cols,
        "prediction_cols": selected_prediction_cols,
        "cumulative_score_cols": cumulative_score_cols,
        "signal_rows": (
            signal_source.height if isinstance(signal_source, pl.DataFrame) else "lazy"
        ),
    }
)


{'feature_return_path': PosixPath('/home/jli/projects/rep/data/orderbook_feature_return_parquet/ESM6_2026-05-28_extra_normal_es_full_day_l2_d5_features_return.parquet'),
 'feature_return_cols': ('forward_mid_return_bps',),
 'prediction_cols': ('prediction_q50',),
 'cumulative_score_cols': ('cumulative_unit_pnl',),
 'signal_rows': 5107233}

## Viewer

The top panel shows bid/ask price levels with L3 trade/add/modify/cancel events overlaid. Events are shown raw when their aggregate count fits the point budget and omitted otherwise. The second panel shows displayed size. Selected feature/return columns, pipeline predictions, and normalized cumulative scores share a third basis-point panel. Scores are calculated from every qualifying row before display subsampling. The pan buttons only move within data loaded for `START`/`END`; rebuild the figure to query another window.


In [9]:
VIEW_KWARGS = dict(
    depth=DEPTH_LEVELS,
    start=START,
    end=END,
    timezone=DISPLAY_TZ,
    product=PRODUCT,
    title=f"{PRODUCT} L2 depth + L3 events | {START} to {END} {DISPLAY_TZ}",
    l3_source=l3,
    l3_depth_filter="levels",
    l3_label_cols=["flags", "order_id", "sequence", "ts_recv"],
    label_cols=["sequence"],
    include_counts=True,
    max_points=MAX_POINTS_PER_SERIES,
    max_event_points=MAX_POINTS_PER_SERIES,
    template="plotly_dark",
    vertical_spacing=0.035,
)

if signal_source is not None:
    VIEW_KWARGS.update(
        signal_source=signal_source,
        signal_cols=selected_signal_cols,
        signal_title=SIGNAL_TITLE,
        signal_yaxis_title=SIGNAL_YAXIS_TITLE,
        signal_value_label=SIGNAL_VALUE_LABEL,
    )

fig = plot_order_book(book, **VIEW_KWARGS, live=True)
controls = add_x_pan_buttons(fig, step=0.5)
display(controls, fig)


FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': '#00875a', 'shape': 'hv', 'width': 1.4},
              'marker': {'color': '#00875a', 'size': 0, 'symbol': 'cross'},
              'mode': 'lines',
              'name': 'bid L1',
              'type': 'scatter',
              'uid': '474a92b8-ff9c-4a44-964a-ff2175ddfb62',
              'x': [0, 7844550, 11472556, ..., 23399731260798, 23399863830532,
                    23399999449674],
              'xaxis': 'x',
              'y': [7535.000000000001, 7535.250000000001, 7535.000000000001, ...,
                    7582.750000000001, 7582.000000000001, 7581.750000000001],
              'yaxis': 'y'},
             {'hoverinfo': 'skip',
              'line': {'color': '#d72638', 'shape': 'hv', 'width': 1.4},
              'marker': {'color': '#d72638', 'size': 0, 'symbol': 'cross'},
              'mode': 'lines',
              'name': 'ask L1',
              'type': 'scatter',
              'uid': '2aaa2d63-

In [10]:
# The left/right buttons pan by half of the currently visible x-window.
# They do not query data outside START/END; rebuild after changing the window.


In [11]:
# fig.show()


### Optional Static Export

In [12]:
# out = PROJECT_ROOT / "notebooks" / f"{PRODUCT}_{SESSION_DATE}_order_book_view.html"
# static_fig = plot_order_book(book, **VIEW_KWARGS)
# static_fig.write_html(out)
# out
